# MINRES: in between GMRES and CG

Elisa Klunder (s5190940) and Dries Wedda (s4745329), 19.01.2026

# Introduction

# MINRES description

# Implementation

In [1]:
import numpy as np
from scipy.linalg import solve_triangular


In [2]:
def make_symmetric_matrix(size: int) -> np.ndarray:
    """Create a symmetric matrix `A` with elements sampled from N(0,1)."""
    A = np.empty((size, size))
    for i in range(size):
        for j in range(i, size):           # all elements j >= i
            A[j, i] = A[i, j] = np.random.normal()
    return A


def make_target(size: int) -> np.ndarray:
    """Create a target vector `b` with elements sampled from N(0,1)."""
    return np.random.normal(size=size)

In [86]:
def lanczos(A: np.ndarray, Q: list, alpha: list, beta: list, j: int) -> list:
    v = A @ Q[j]
    alpha.append(np.dot(v, Q[j]))
    v = v - alpha[j] * Q[j] - beta[j - 1] * Q[j - 1]
    beta.append(np.linalg.norm(v))
    Q.append(v / beta[j])    # q_j = v / beta_j


def make_hessenberg(alpha: list, beta: list, k: int) -> np.ndarray:
    H = np.zeros((k + 1, k))
    H[0, 0] = alpha[0]
    H[1, 0] = beta[0]
    for idx in range(1, k):
        H[idx, idx] = alpha[idx]
        H[idx - 1, idx] = beta[idx - 1]
        H[idx + 1, idx] = beta[idx]
    return H

def minres(A: np.ndarray, b: np.ndarray, max_iter: int, tolerance: float = 1e-12):
    # TODO list or np array for alpha and beta?

    Q = [b / np.linalg.norm(b)]
    
    v = A @ Q[0]
    alpha = [np.dot(v, Q[0])]     # alpha_1 = v.T q_1
    v -= alpha[0] * Q[0]
    beta = [np.linalg.norm(v)]    # beta_1 = ||v||
    Q.append(v / beta[0])         # q_2 = v / beta_1
    
    for k in range(1, max_iter + 1):
        # print(f"k = {k}")

        H = make_hessenberg(alpha, beta, k)
        V, R = np.linalg.qr(H)                  # TODO add Givens rotations

        z = np.zeros(k + 1)                     # ||b||*e_1 for R^{k+1}
        z[0] = np.linalg.norm(b)

        y = solve_triangular(R, V.T @ z, lower=False)   # y = R^-1 V.T z --> Ry = V.T z
                                                        # note R is upper triangular!
        x = np.array(Q).T[:,:k] @ y
        
        residual = np.linalg.norm(A @ x - b)
        if residual < tolerance:
            print("tolerance reached")
            return x

        if k < max_iter:
            lanczos(A, Q, alpha, beta, k)
    return x

np.random.seed(0)
m = 16
A = make_symmetric_matrix(m)
b = make_target(m)
x = minres(A, b, max_iter=b.shape[0])

x_direct = np.linalg.solve(A, b)
print("Direct solve residual:", np.linalg.norm(A @ x_direct - b))
print("residual:", np.linalg.norm(A @ x - b))

Direct solve residual: 8.555349471030667e-15
residual: 1.6873739764554356e-10


# Comparison

# Extension